# 01 Prepare S3PO Splits (Unified)

Goal:
- Rebuild S3PO-derived datasets under `dataset/*/part2_s3po/{sparse,test}`.
- Use equal-interval sampling for sparse split and put the remainder into test split.
- Keep split folders directly readable by S3PO loaders.
- Generate sparse/test configs and preparation checks.

Input policy in this notebook:
- `Re10k-1` and `DL3DV-2` use `part1/shared/images_512` + `part1/shared/meta/*.json`.
- `405841` uses `part1/shared/images_512` for RGB and `FRONT/depth`, `FRONT/gt` for depth/pose.

Sampling policy in this notebook:
- `Re10k-1`: sparse count = 9 (about 1/30).
- `DL3DV-2`: sparse count = 10 (about 1/30).
- `405841`: sparse count = 20 (about 1/10), and use Waymo loader (no scene split).

Note for Waymo synchronization:
- Waymo loader requires `mono_depth/*.png`; `None` cannot be parsed.
- For `405841`, this notebook resizes depth to RGB-512 shape and symlinks `mono_depth -> depth`.
- Waymo calibration is scaled from official 1920x1280 values to current RGB size and distortion is disabled for resized data.

In [1]:
from pathlib import Path
import json
import csv
import shutil
from datetime import datetime

import numpy as np
from PIL import Image
import yaml

PROJECT_ROOT = Path('/home/bzhang512/CV_Project')
S3PO_ROOT = PROJECT_ROOT / 'third_party' / 'S3PO-GS'
CONFIG_DIR = PROJECT_ROOT / 'part2_s3po/configs'
CHECK_DIR = PROJECT_ROOT / 'part2_s3po/checks'

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CHECK_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('S3PO_ROOT =', S3PO_ROOT)
print('CONFIG_DIR =', CONFIG_DIR)
print('CHECK_DIR =', CHECK_DIR)

PROJECT_ROOT = /home/bzhang512/CV_Project
S3PO_ROOT = /home/bzhang512/CV_Project/third_party/S3PO-GS
CONFIG_DIR = /home/bzhang512/CV_Project/part2_s3po/configs
CHECK_DIR = /home/bzhang512/CV_Project/part2_s3po/checks


In [2]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=True)


def list_pngs(folder: Path):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == '.png'])


def safe_reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def intrinsics_norm_to_px(norm_intrinsics: dict, width: int, height: int):
    return {
        'fx': float(norm_intrinsics['fx']) * float(width),
        'fy': float(norm_intrinsics['fy']) * float(height),
        'cx': float(norm_intrinsics['cx']) * float(width),
        'cy': float(norm_intrinsics['cy']) * float(height),
    }


def equal_interval_indices(n_total: int, n_sparse: int):
    if n_sparse <= 0:
        raise ValueError(f'n_sparse must be > 0, got {n_sparse}')
    if n_sparse > n_total:
        raise ValueError(f'n_sparse ({n_sparse}) exceeds n_total ({n_total})')
    ids = np.linspace(0, n_total - 1, n_sparse).astype(int).tolist()
    ids = sorted(set(ids))
    while len(ids) < n_sparse:
        for x in range(n_total):
            if x not in ids:
                ids.append(x)
            if len(ids) == n_sparse:
                break
        ids = sorted(ids)
    return ids


def symlink_files(src_dir: Path, dst_dir: Path, names):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for name in names:
        src = src_dir / name
        if not src.exists():
            raise FileNotFoundError(f'Missing source file: {src}')
        dst = dst_dir / name
        dst.symlink_to(src)


def write_csv(path: Path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

## Prepare Re10k-1 and DL3DV-2 (dl3dv loader format)

In [3]:
dl3dv_like_specs = [
    {
        'dataset_name': 'Re10k-1',
        'scene_key': 're10k1',
        'source_root': PROJECT_ROOT / 'dataset/Re10k-1/part1/shared',
        'target_root': PROJECT_ROOT / 'dataset/Re10k-1/part2_s3po',
        'n_sparse': 9,
        'loader_type': 'dl3dv',
    },
    {
        'dataset_name': 'DL3DV-2',
        'scene_key': 'dl3dv2',
        'source_root': PROJECT_ROOT / 'dataset/DL3DV-2/part1/shared',
        'target_root': PROJECT_ROOT / 'dataset/DL3DV-2/part2_s3po',
        'n_sparse': 10,
        'loader_type': 'dl3dv',
    },
]

dl3dv_rows = []

for spec in dl3dv_like_specs:
    source_root = spec['source_root']
    target_root = spec['target_root']

    src_images = source_root / 'images_512'
    src_cameras = source_root / 'meta/cameras.json'
    src_intr = source_root / 'meta/intrinsics.json'

    assert source_root.exists(), f'Missing source_root: {source_root}'
    assert src_images.exists(), f'Missing images dir: {src_images}'
    assert src_cameras.exists(), f'Missing cameras: {src_cameras}'
    assert src_intr.exists(), f'Missing intrinsics: {src_intr}'

    image_paths = list_pngs(src_images)
    image_names = [p.name for p in image_paths]
    n_total = len(image_names)
    assert n_total > 0, f'No pngs in {src_images}'

    cams = read_json(src_cameras)
    assert len(cams) == n_total, f'camera/image mismatch for {spec["dataset_name"]}'
    cam_by_name = {c['image_name']: c for c in cams}
    if set(cam_by_name.keys()) != set(image_names):
        raise ValueError(f'image_name mismatch in {spec["dataset_name"]}')

    sparse_ids = equal_interval_indices(n_total, spec['n_sparse'])
    sparse_set = set(sparse_ids)
    test_ids = [i for i in range(n_total) if i not in sparse_set]

    split_map = {
        'sparse': sparse_ids,
        'test': test_ids,
    }

    width, height = Image.open(image_paths[0]).size
    intr_norm = read_json(src_intr)
    intr_px = intrinsics_norm_to_px(intr_norm, width, height)

    for split_name, ids in split_map.items():
        split_root = target_root / split_name
        safe_reset_dir(split_root)

        selected_names = [image_names[i] for i in ids]
        selected_cams = []
        for local_id, name in enumerate(selected_names):
            c = dict(cam_by_name[name])
            c['cam_id'] = local_id
            selected_cams.append(c)

        symlink_files(src_images, split_root / 'rgb', selected_names)
        write_json(split_root / 'cameras.json', selected_cams)

        (split_root / 'intrinsics_norm.json').symlink_to(src_intr)
        write_json(split_root / 'intrinsics_px.json', intr_px)

        manifest = {
            'dataset_name': spec['dataset_name'],
            'split': split_name,
            'loader_type': 'dl3dv',
            'created_at': datetime.now().isoformat(timespec='seconds'),
            'n_total': n_total,
            'n_split': len(ids),
            'sparse_count_policy': spec['n_sparse'],
            'selected_indices': ids,
            'width': width,
            'height': height,
            'files': {
                'rgb': 'rgb',
                'cameras': 'cameras.json',
                'intrinsics_norm': 'intrinsics_norm.json',
                'intrinsics_px': 'intrinsics_px.json',
            },
        }
        write_json(split_root / 'split_manifest.json', manifest)

        cfg = yaml.safe_load((S3PO_ROOT / 'configs/mono/dl3dv/base_config.yaml').read_text(encoding='utf-8'))
        cfg['Dataset']['type'] = 'dl3dv'
        cfg['Dataset']['sensor_type'] = 'monocular'
        cfg['Dataset']['dataset_path'] = str(split_root)
        cfg['Dataset']['begin'] = 0
        cfg['Dataset']['end'] = len(ids)
        cfg['Dataset']['Calibration'] = {
            'fx': float(intr_px['fx']),
            'fy': float(intr_px['fy']),
            'cx': float(intr_px['cx']),
            'cy': float(intr_px['cy']),
            'k1': 0.0,
            'k2': 0.0,
            'p1': 0.0,
            'p2': 0.0,
            'k3': 0.0,
            'width': int(width),
            'height': int(height),
            'depth_scale': 200.0,
            'distorted': False,
        }

        cfg_name = f's3po_{spec["scene_key"]}_{split_name}.yaml'
        cfg_path = CONFIG_DIR / cfg_name
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

        dl3dv_rows.append({
            'dataset_name': spec['dataset_name'],
            'split': split_name,
            'loader_type': 'dl3dv',
            'n_total': n_total,
            'n_split': len(ids),
            'target_root': str(split_root),
            'config_path': str(cfg_path),
        })

    print(f"{spec['dataset_name']} done: total={n_total}, sparse={len(sparse_ids)}, test={len(test_ids)}")

Re10k-1 done: total=279, sparse=9, test=270
DL3DV-2 done: total=306, sparse=10, test=296


## Prepare 405841 (waymo loader format, no scene split)

In [4]:
WAYMO_RGB_SRC = PROJECT_ROOT / 'dataset/405841/part1/shared/images_512'
WAYMO_DEPTH_SRC = PROJECT_ROOT / 'dataset/405841/FRONT/depth'
WAYMO_GT_SRC = PROJECT_ROOT / 'dataset/405841/FRONT/gt'
WAYMO_DST = PROJECT_ROOT / 'dataset/405841/part2_s3po'
WAYMO_SPARSE_COUNT = 20

assert WAYMO_RGB_SRC.exists(), f'Missing waymo RGB source: {WAYMO_RGB_SRC}'
assert WAYMO_DEPTH_SRC.exists(), f'Missing waymo depth source: {WAYMO_DEPTH_SRC}'
assert WAYMO_GT_SRC.exists(), f'Missing waymo gt source: {WAYMO_GT_SRC}'

rgb_names = sorted([p.name for p in WAYMO_RGB_SRC.glob('*.png')])
depth_names = sorted([p.name for p in WAYMO_DEPTH_SRC.glob('*.png')])
gt_names = sorted([p.name for p in WAYMO_GT_SRC.glob('*.txt')])

rgb_stems = {Path(x).stem for x in rgb_names}
depth_stems = {Path(x).stem for x in depth_names}
gt_stems = {Path(x).stem for x in gt_names}
common_stems = sorted(list(rgb_stems & depth_stems & gt_stems))
assert len(common_stems) > 0, 'No common stem among waymo rgb/depth/gt'

n_total = len(common_stems)
sparse_ids = equal_interval_indices(n_total, WAYMO_SPARSE_COUNT)
sparse_set = set(sparse_ids)
test_ids = [i for i in range(n_total) if i not in sparse_set]

split_map = {
    'sparse': sparse_ids,
    'test': test_ids,
}

waymo_cfg_file = S3PO_ROOT / 'configs/mono/waymo/405841.yaml'
waymo_cfg = yaml.safe_load(waymo_cfg_file.read_text(encoding='utf-8'))
waymo_calib_raw = dict(waymo_cfg['Dataset']['Calibration'])

rgb_sample = Image.open(WAYMO_RGB_SRC / rgb_names[0])
target_width, target_height = rgb_sample.size
rgb_sample.close()

src_width = float(waymo_calib_raw['width'])
src_height = float(waymo_calib_raw['height'])
sx = float(target_width) / src_width
sy = float(target_height) / src_height

waymo_calib_sync = {
    'fx': float(waymo_calib_raw['fx']) * sx,
    'fy': float(waymo_calib_raw['fy']) * sy,
    'cx': float(waymo_calib_raw['cx']) * sx,
    'cy': float(waymo_calib_raw['cy']) * sy,
    'k1': 0.0,
    'k2': 0.0,
    'p1': 0.0,
    'p2': 0.0,
    'k3': 0.0,
    'width': int(target_width),
    'height': int(target_height),
    'depth_scale': float(waymo_calib_raw.get('depth_scale', 5000.0)),
    'distorted': False,
}

waymo_rows = []

for split_name, ids in split_map.items():
    split_root = WAYMO_DST / split_name
    safe_reset_dir(split_root)

    for sub in ['rgb', 'depth', 'mono_depth', 'gt']:
        (split_root / sub).mkdir(parents=True, exist_ok=True)

    selected_stems = [common_stems[i] for i in ids]

    for stem in selected_stems:
        rgb_name = f'{stem}.png'
        gt_name = f'{stem}.txt'

        # RGB already resized to 512 in part1/shared/images_512.
        (split_root / 'rgb' / rgb_name).symlink_to(WAYMO_RGB_SRC / rgb_name)

        # Resize depth to match RGB-512 shape and save as PNG in split/depth.
        depth_src = WAYMO_DEPTH_SRC / rgb_name
        depth_dst = split_root / 'depth' / rgb_name
        with Image.open(depth_src) as dep_img:
            dep_512 = dep_img.resize((target_width, target_height), Image.NEAREST)
            dep_512.save(depth_dst)

        # waymo loader requires mono_depth png files; keep it aligned with resized depth.
        (split_root / 'mono_depth' / rgb_name).symlink_to(depth_dst)
        (split_root / 'gt' / gt_name).symlink_to(WAYMO_GT_SRC / gt_name)

    manifest = {
        'dataset_name': '405841',
        'split': split_name,
        'loader_type': 'waymo',
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'n_total': n_total,
        'n_split': len(ids),
        'sparse_count_policy': WAYMO_SPARSE_COUNT,
        'selected_indices': ids,
        'selected_stems': selected_stems,
        'rgb_source': str(WAYMO_RGB_SRC),
        'depth_source': str(WAYMO_DEPTH_SRC),
        'gt_source': str(WAYMO_GT_SRC),
        'depth_resize': {
            'source_size': [int(src_width), int(src_height)],
            'target_size': [int(target_width), int(target_height)],
            'resample': 'NEAREST',
        },
        'calibration_sync': waymo_calib_sync,
        'mono_depth_placeholder': 'symlink_to_resized_depth',
        'files': {
            'rgb': 'rgb',
            'depth': 'depth',
            'mono_depth': 'mono_depth',
            'gt': 'gt',
        },
    }
    write_json(split_root / 'split_manifest.json', manifest)

    cfg = yaml.safe_load((S3PO_ROOT / 'configs/mono/waymo/base_config.yaml').read_text(encoding='utf-8'))
    cfg['Dataset']['type'] = 'waymo'
    cfg['Dataset']['sensor_type'] = 'monocular'
    cfg['Dataset']['dataset_path'] = str(split_root)
    cfg['Dataset']['Calibration'] = waymo_calib_sync

    cfg_name = f's3po_405841_{split_name}_waymo.yaml'
    cfg_path = CONFIG_DIR / cfg_name
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

    waymo_rows.append({
        'dataset_name': '405841',
        'split': split_name,
        'loader_type': 'waymo',
        'n_total': n_total,
        'n_split': len(ids),
        'target_root': str(split_root),
        'config_path': str(cfg_path),
    })

print(f"405841 done: total={n_total}, sparse={len(sparse_ids)}, test={len(test_ids)}")
print('405841 synced calibration =', waymo_calib_sync)

405841 done: total=199, sparse=20, test=179
405841 synced calibration = {'fx': 551.1193505112797, 'fy': 826.6790257669196, 'cx': 253.48034064401926, 'cy': 256.4748216588868, 'k1': 0.0, 'k2': 0.0, 'p1': 0.0, 'p2': 0.0, 'k3': 0.0, 'width': 512, 'height': 512, 'depth_scale': 5000.0, 'distorted': False}


## Validate outputs and write summary checks

In [5]:
summary_rows = dl3dv_rows + waymo_rows

for row in summary_rows:
    root = Path(row['target_root'])
    split = row['split']
    loader_type = row['loader_type']

    if loader_type == 'dl3dv':
        rgb = list_pngs(root / 'rgb')
        cams = read_json(root / 'cameras.json')
        assert len(rgb) == row['n_split'], f"{root}: rgb count mismatch"
        assert len(cams) == row['n_split'], f"{root}: camera count mismatch"
        names_rgb = {p.name for p in rgb}
        names_cam = {c['image_name'] for c in cams}
        assert names_rgb == names_cam, f"{root}: image_name mismatch"
        assert (root / 'intrinsics_norm.json').is_symlink(), f"{root}: intrinsics_norm.json should be symlink"
        assert (root / 'intrinsics_px.json').exists(), f"{root}: missing intrinsics_px.json"

        if rgb:
            with Image.open(rgb[0]) as im:
                w, h = im.size
            intr_px = read_json(root / 'intrinsics_px.json')
            assert int(round(intr_px['cx'] * 2.0)) == int(w), f"{root}: cx not centered for width {w}"
            assert int(round(intr_px['cy'] * 2.0)) == int(h), f"{root}: cy not centered for height {h}"
    else:
        rgb = list_pngs(root / 'rgb')
        dep = list_pngs(root / 'depth')
        mono = list_pngs(root / 'mono_depth')
        gt = sorted([p for p in (root / 'gt').iterdir() if p.is_file() and p.suffix.lower() == '.txt'])
        assert len(rgb) == row['n_split'], f"{root}: rgb count mismatch"
        assert len(dep) == row['n_split'], f"{root}: depth count mismatch"
        assert len(mono) == row['n_split'], f"{root}: mono_depth count mismatch"
        assert len(gt) == row['n_split'], f"{root}: gt count mismatch"

        if rgb:
            with Image.open(rgb[0]) as im_rgb:
                rgb_size = im_rgb.size
            with Image.open(dep[0]) as im_dep:
                dep_size = im_dep.size
                dep_mode = im_dep.mode
            with Image.open(mono[0]) as im_mono:
                mono_size = im_mono.size
            assert rgb_size == dep_size == mono_size, f"{root}: rgb/depth/mono size mismatch"
            assert rgb_size == (512, 512), f"{root}: expected waymo split size (512, 512), got {rgb_size}"
            assert dep_mode in {'I;16', 'I', 'L'}, f"{root}: unexpected depth mode {dep_mode}"

    row['check_pass'] = True

summary_json = CHECK_DIR / 'part2_s3po_split_prepare_summary.json'
summary_csv = CHECK_DIR / 'part2_s3po_split_prepare_summary.csv'
write_json(summary_json, {'rows': summary_rows})
write_csv(summary_csv, summary_rows, fieldnames=['dataset_name', 'split', 'loader_type', 'n_total', 'n_split', 'target_root', 'config_path', 'check_pass'])

print('Wrote summary JSON:', summary_json)
print('Wrote summary CSV:', summary_csv)
for row in summary_rows:
    print(row['dataset_name'], row['split'], 'n_split=', row['n_split'], 'loader=', row['loader_type'])

Wrote summary JSON: /home/bzhang512/CV_Project/part2_s3po/checks/part2_s3po_split_prepare_summary.json
Wrote summary CSV: /home/bzhang512/CV_Project/part2_s3po/checks/part2_s3po_split_prepare_summary.csv
Re10k-1 sparse n_split= 9 loader= dl3dv
Re10k-1 test n_split= 270 loader= dl3dv
DL3DV-2 sparse n_split= 10 loader= dl3dv
DL3DV-2 test n_split= 296 loader= dl3dv
405841 sparse n_split= 20 loader= waymo
405841 test n_split= 179 loader= waymo
